# 🧠 Building a Language Model From Scratch

Welcome! This notebook walks through **every step** of building and training a GPT-style language model — the same architecture family behind ChatGPT, GPT-4, and LLaMA.

By the end, you'll understand:

1. **Tokenization** — How text becomes numbers (and why it's not trivial)
2. **Embeddings** — How numbers become meaningful vectors
3. **Self-Attention** — The key innovation that makes transformers work (we'll build it from scratch!)
4. **The Full Model** — How all the pieces fit together
5. **Training** — How the model learns patterns from text
6. **Generation** — How to make the model write new text

### Prerequisites
- Basic Python knowledge
- Some familiarity with NumPy/PyTorch tensors (we'll explain shapes as we go)
- No prior knowledge of transformers needed!

### How to Read This Notebook
Each section starts with an **intuitive explanation** before showing code. We'll use small, concrete examples so you can follow the math by hand.

---

In [ ]:
# === Setup: Import libraries ===
# We need to add the parent directory to Python's path so we can import
# our project modules (model.py, config.py, etc.)
import sys
sys.path.insert(0, '..')  # '..' means 'parent directory'

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math

# Set a random seed for reproducibility
# (so you get the same numbers every time you run this notebook)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("\n✅ Setup complete! Let's build a language model.")

---
## 1. 📝 Tokenization: From Text to Numbers

### The Problem
Neural networks can only work with numbers — they can't process text directly. We need a way to convert text like `"Hello world"` into numbers like `[42, 100]`.

### Three Approaches (and why BPE wins)

| Approach | Example: "understanding" | Vocab Size | Pros | Cons |
|----------|-------------------------|------------|------|------|
| **Character-level** | `['u','n','d','e','r','s','t','a','n','d','i','n','g']` | ~100 | Can handle any text | Sequences very long, hard to learn |
| **Word-level** | `['understanding']` | ~1,000,000 | Meaningful units | Can't handle new words, huge vocab |
| **BPE (sub-word)** | `['under', 'stand', 'ing']` | 4,000-50,000 | Best of both worlds ✅ | Slightly complex to implement |

We use **BPE (Byte-Pair Encoding)**, the same algorithm used by GPT-2, GPT-3, and most modern LLMs.

### How BPE Works (Intuition)
1. Start with individual characters: `{a, b, c, ..., z}`
2. Find the most frequent pair of adjacent tokens in your text
3. Merge them into a new token: e.g., `'t' + 'h'` → `'th'`
4. Repeat until you reach your desired vocabulary size

After enough merges, common words become single tokens (`"the"`, `"and"`), while rare words get split into recognizable pieces (`"brontosaurus"` → `"br" + "onto" + "saur" + "us"`).

In [ ]:
from train_tokenizer import load_tokenizer

tokenizer = load_tokenizer()
print(f"Vocabulary size: {tokenizer.get_vocab_size()} tokens")
print(f"(For comparison: GPT-2 has 50,257 tokens, LLaMA has 32,000)")

# === Let's see tokenization in action! ===
text = "Once upon a time, there was a little girl named Lily."
encoded = tokenizer.encode(text)

print(f"\n{'='*60}")
print(f"Original text: {text}")
print(f"{'='*60}")
print(f"\nTokens (sub-words): {encoded.tokens}")
print(f"Token IDs (integers): {encoded.ids}")
print(f"Number of tokens: {len(encoded.ids)}")
print(f"\nDecoded back: {tokenizer.decode(encoded.ids)}")

# Show the mapping token-by-token
print(f"\n--- Token-by-Token Mapping ---")
for token, id_ in zip(encoded.tokens, encoded.ids):
    print(f"  '{token:15s}' → ID {id_}")

print(f"\n💡 Notice: <BOS> (Beginning Of Sequence) and <EOS> (End Of Sequence)")
print(f"   are automatically added by the tokenizer.")
print(f"   These tell the model where text starts and ends.")

In [ ]:
# === How does BPE handle rare/unknown words? ===
print("--- BPE with Rare Words ---\n")

examples = [
    "The cat sat on the mat",          # Common words → single tokens
    "The brontosaurus ate phenomenally", # Rare words → split into sub-words
    "Python programming is fun",        # Technical words
    "Supercalifragilistic",             # Very long/unusual word
]

for text in examples:
    encoded = tokenizer.encode(text)
    # Filter out <BOS> and <EOS> for cleaner display
    tokens = [t for t in encoded.tokens if t not in ('<BOS>', '<EOS>')]
    print(f"  Text:   '{text}'")
    print(f"  Tokens: {tokens}")
    print(f"  Count:  {len(tokens)} tokens")
    print()

---
## 2. 🔢 Embeddings: From Numbers to Meaning

### The Problem with Raw Token IDs
Token IDs are just arbitrary integers. The model needs to know that:
- `"cat"` (ID 42) and `"dog"` (ID 78) are similar (both animals)
- `"cat"` (ID 42) and `"quantum"` (ID 200) are not similar

But mathematically, 42 is just as far from 78 as it is from 200!

### The Solution: Embedding Vectors
We map each token ID to a **learned vector** of `d_model` numbers (128 in our case). During training, the model adjusts these vectors so that similar words end up near each other in this high-dimensional space.

```
Token ID 42 ("cat") → [0.12, -0.03, 0.55, ..., 0.01]   (128 numbers)
Token ID 78 ("dog") → [0.15, -0.01, 0.52, ..., -0.02]  (128 numbers, similar to cat!)
Token ID 200 ("quantum") → [-0.32, 0.44, -0.11, ..., 0.38] (128 numbers, very different)
```

### Why Add Positional Embeddings?
Unlike RNNs that process tokens one-by-one, transformers see all tokens at once. Without position information, `"the cat ate the fish"` looks the same as `"the fish ate the cat"` — the same tokens, just in different order!

Positional embeddings add a unique vector for each position (0, 1, 2, ...), so the model knows word order.

In [ ]:
from config import ModelConfig
cfg = ModelConfig()

print("=== Model Configuration ===")
print(f"  vocab_size:   {cfg.vocab_size:,} tokens (size of our dictionary)")
print(f"  d_model:      {cfg.d_model} dimensions (width of each embedding vector)")
print(f"  max_seq_len:  {cfg.max_seq_len} tokens (maximum context the model can see)")
print(f"  n_layers:     {cfg.n_layers} transformer blocks (depth of the model)")
print(f"  n_heads:      {cfg.n_heads} attention heads (parallel attention patterns)")
print(f"  d_ff:         {cfg.d_ff} (feed-forward inner dimension = 4 × d_model)")

print(f"\n=== Embedding Demo ===")

# Create a token embedding table: vocab_size × d_model = 4000 × 128
token_emb = torch.nn.Embedding(cfg.vocab_size, cfg.d_model)
print(f"\nEmbedding table shape: {token_emb.weight.shape}")
print(f"  → {cfg.vocab_size} rows (one per token in vocabulary)")
print(f"  → {cfg.d_model} columns (embedding dimension)")
print(f"  → Total: {cfg.vocab_size * cfg.d_model:,} learnable numbers")

# Embed a short sequence
sample_ids = torch.tensor([[42, 100, 200, 5]])  # batch=1, seq_len=4
embedded = token_emb(sample_ids)

print(f"\nInput token IDs: {sample_ids.tolist()}")
print(f"  Shape: {sample_ids.shape}  (batch=1, seq_len=4)")
print(f"\nAfter embedding lookup:")
print(f"  Shape: {embedded.shape}  (batch=1, seq_len=4, d_model={cfg.d_model})")
print(f"\n  Token 42 → vector: [{embedded[0, 0, :5].tolist()}... ]  (showing first 5 of {cfg.d_model})")
print(f"  Token 100 → vector: [{embedded[0, 1, :5].tolist()}... ]")
print(f"\n💡 Each token ID becomes a {cfg.d_model}-dimensional vector!")
print(f"   These vectors are RANDOM at first, but get refined during training.")

In [ ]:
# === Visualize what embeddings look like ===
# Let's look at the embedding vectors for a few tokens

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot 1: Raw embedding vectors for 4 tokens
token_ids_to_show = [42, 100, 200, 5]
for i, tid in enumerate(token_ids_to_show):
    vec = token_emb.weight[tid].detach().numpy()
    axes[0].plot(vec[:32], label=f'Token {tid}', alpha=0.7)  # Show first 32 dims
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Value')
axes[0].set_title('Embedding Vectors (first 32 dims)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Embedding similarity matrix
# Compute cosine similarity between some token embeddings
n_tokens = 20
emb_subset = token_emb.weight[:n_tokens].detach()  # First 20 tokens
# Normalize embeddings
emb_norm = emb_subset / emb_subset.norm(dim=-1, keepdim=True)
# Cosine similarity matrix
sim_matrix = emb_norm @ emb_norm.T
im = axes[1].imshow(sim_matrix.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title(f'Cosine Similarity (first {n_tokens} tokens)')
axes[1].set_xlabel('Token ID'); axes[1].set_ylabel('Token ID')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

print("💡 At initialization, embeddings are random (no meaningful similarity).")
print("   After training, similar tokens will have similar vectors!")

---
## 3. 🎯 Self-Attention: The Heart of the Transformer

This is the most important concept to understand! Let's build it step by step.

### The Big Idea
When reading a sentence, each word needs **context** from other words to be understood:

> *"The **cat** sat on the mat because **it** was tired."*

To understand `"it"`, we need to look back at `"cat"` (the referent). Self-attention is the mechanism that lets each token **look at other tokens** and decide how much to pay attention to each one.

### The Q, K, V Framework

Self-attention uses three vectors per token:

| Vector | Name | Analogy | Purpose |
|--------|------|---------|--------|
| **Q** | Query | "What am I looking for?" | What info this token needs |
| **K** | Key | "What do I contain?" | What info this token offers |
| **V** | Value | "Here's my content" | The actual info to pass along |

Think of it like a search engine:
- Each token broadcasts a **Key** ("I'm a noun", "I'm a verb", etc.)
- Each token has a **Query** ("I need to find the subject", etc.)
- The **dot product Q·K** measures how well a query matches a key
- High match → pay more attention → use more of that token's **Value**

Let's implement this step by step with actual numbers!

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║         SELF-ATTENTION: STEP-BY-STEP WITH REAL NUMBERS         ║
# ╚══════════════════════════════════════════════════════════════════╝

# Let's use a TINY example so we can follow every number:
# 4 tokens, each with d_head=3 dimensions (real models use 32-128)

seq_len = 4       # 4 tokens: ["The", "cat", "sat", "on"]
d_head = 3        # 3 dimensions per head (tiny, for illustration)

# === Step 0: Start with token representations ===
# Normally these come from the embedding layer. Let's use small numbers.
torch.manual_seed(42)

# In reality, Q, K, V come from linear projections of the input:
#   Q = input @ W_Q    (what each token is looking for)
#   K = input @ W_K    (what each token offers)
#   V = input @ W_V    (each token's content to share)
# For this demo, we'll use random Q, K, V directly.

Q = torch.tensor([[ 1.0,  0.5,  0.0],   # Query for "The"
                   [ 0.0,  1.0,  0.5],   # Query for "cat"
                   [ 0.5,  0.0,  1.0],   # Query for "sat"
                   [ 0.3,  0.3,  0.3]])  # Query for "on"

K = torch.tensor([[ 0.5,  0.5,  0.0],   # Key for "The"
                   [ 0.0,  0.8,  0.2],   # Key for "cat"
                   [ 0.7,  0.0,  0.3],   # Key for "sat"
                   [ 0.2,  0.2,  0.6]])  # Key for "on"

V = torch.tensor([[ 1.0,  0.0,  0.0],   # Value for "The"
                   [ 0.0,  1.0,  0.0],   # Value for "cat"
                   [ 0.0,  0.0,  1.0],   # Value for "sat"
                   [ 0.5,  0.5,  0.5]])  # Value for "on"

token_names = ["The", "cat", "sat", "on"]

print("=== Our tiny example ===")
print(f"Sequence: {token_names}")
print(f"Q (Queries) shape: {Q.shape}  — what each token is looking for")
print(f"K (Keys) shape:    {K.shape}  — what each token offers")
print(f"V (Values) shape:  {V.shape}  — each token's content")
print()
for i, name in enumerate(token_names):
    print(f"  Token '{name}': Q={Q[i].tolist()}, K={K[i].tolist()}, V={V[i].tolist()}")

In [ ]:
# === Step 1: Compute attention scores (Q · K^T) ===
# The score between token i and token j is the dot product of Q[i] and K[j].
# High score = "token i should pay attention to token j"

print("=" * 60)
print("STEP 1: Compute attention scores (Q · K^T)")
print("=" * 60)

# Manual computation for one pair:
# score("cat", "The") = Q["cat"] · K["The"]
#                      = [0.0, 1.0, 0.5] · [0.5, 0.5, 0.0]
#                      = 0.0*0.5 + 1.0*0.5 + 0.5*0.0
#                      = 0.5
manual_score = (Q[1] * K[0]).sum()
print(f"\nManual: score('cat' → 'The') = Q_cat · K_The")
print(f"  = {Q[1].tolist()} · {K[0].tolist()}")
print(f"  = {Q[1][0]:.1f}×{K[0][0]:.1f} + {Q[1][1]:.1f}×{K[0][1]:.1f} + {Q[1][2]:.1f}×{K[0][2]:.1f}")
print(f"  = {manual_score:.2f}")

# Now compute ALL scores at once with matrix multiplication
raw_scores = Q @ K.T  # (4, 3) @ (3, 4) → (4, 4)

print(f"\nFull score matrix (Q @ K^T):")
print(f"  Shape: {raw_scores.shape} — one score for each (query, key) pair")
print()

# Pretty-print the score matrix
print(f"{'':>10}", end="")
for name in token_names:
    print(f"{name:>8}", end="")
print("  ← Key (being attended TO)")
for i, name in enumerate(token_names):
    print(f"  {name:>6}  ", end="")
    for j in range(seq_len):
        print(f"{raw_scores[i, j]:8.2f}", end="")
    print()
print("  ↑ Query (doing the attending)")

In [ ]:
# === Step 2: Scale by √d_head ===
# WHY? The dot product grows with dimension. Without scaling, large scores
# would make softmax nearly one-hot (only attending to one token).

print("=" * 60)
print("STEP 2: Scale by 1/√d_head")
print("=" * 60)

scale = math.sqrt(d_head)  # √3 ≈ 1.73
print(f"\nd_head = {d_head}, so √d_head = √{d_head} = {scale:.2f}")
print(f"We divide all scores by {scale:.2f} to prevent softmax from becoming too peaked.")

scaled_scores = raw_scores / scale

print(f"\nBefore scaling: raw_scores[0] = {raw_scores[0].tolist()}")
print(f"After scaling:  scaled_scores[0] = [{', '.join(f'{x:.3f}' for x in scaled_scores[0].tolist())}]")

# Demonstrate why scaling matters with a larger example
print(f"\n--- Why Scaling Matters ---")
big_scores = torch.tensor([10.0, 1.0, 0.5])
small_scores = big_scores / math.sqrt(32)  # As if d_head=32
print(f"Without scaling: softmax({big_scores.tolist()}) = {F.softmax(big_scores, dim=-1).tolist()}")
print(f"  → Almost all weight on first token! (one-hot-like)")
print(f"With scaling (÷√32): softmax({[f'{x:.2f}' for x in small_scores.tolist()]}) = "
      f"{[f'{x:.3f}' for x in F.softmax(small_scores, dim=-1).tolist()]}")
print(f"  → More balanced attention across tokens ✅")

In [ ]:
# === Step 3: Apply the CAUSAL MASK ===
# This is what makes it "causal" (autoregressive) — each token can only
# attend to tokens that came BEFORE it (and itself). No peeking ahead!

print("=" * 60)
print("STEP 3: Apply the Causal Mask")
print("=" * 60)

# Create the causal mask: True = "this position should be blocked"
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

print(f"\nCausal mask (True = BLOCKED):")
for i, name in enumerate(token_names):
    row = ""
    for j in range(seq_len):
        if causal_mask[i, j]:
            row += "  BLOCK"
        else:
            row += "  allow"
    print(f"  {name:>4}: {row}")
print(f"          {'The':>6} {'cat':>6} {'sat':>6} {'on':>6}")

print(f"\nInterpretation:")
print(f"  'The' (pos 0) can only see: itself")
print(f"  'cat' (pos 1) can see: 'The', itself")
print(f"  'sat' (pos 2) can see: 'The', 'cat', itself")
print(f"  'on'  (pos 3) can see: 'The', 'cat', 'sat', itself")

# Apply the mask: set blocked positions to -infinity
masked_scores = scaled_scores.clone()
masked_scores[causal_mask] = float('-inf')

print(f"\nScaled scores BEFORE mask:")
for i, name in enumerate(token_names):
    print(f"  {name:>4}: [{', '.join(f'{x:6.2f}' for x in scaled_scores[i].tolist())}]")

print(f"\nScaled scores AFTER mask (future positions → -inf):")
for i, name in enumerate(token_names):
    print(f"  {name:>4}: [{', '.join(f'{x:6.2f}' if x != float('-inf') else '  -inf' for x in masked_scores[i].tolist())}]")

print(f"\n💡 -inf positions will become 0 after softmax (next step).")
print(f"   This prevents any information leaking from the future!")

In [ ]:
# === Step 4: Softmax → Attention Weights ===
# Convert raw scores to probabilities. Each row sums to 1.
# Higher score → higher probability → more attention paid.

print("=" * 60)
print("STEP 4: Softmax → Attention Weights")
print("=" * 60)

attn_weights = F.softmax(masked_scores, dim=-1)

print(f"\nSoftmax formula: weight[i,j] = exp(score[i,j]) / Σ_k exp(score[i,k])")
print(f"After softmax, each row sums to 1.0 (it's a probability distribution).")
print(f"Tokens with -inf scores get weight 0 (blocked by causal mask).")

print(f"\nAttention weights (who pays attention to whom):")
print(f"{'':>10}", end="")
for name in token_names:
    print(f"{name:>8}", end="")
print(f"  {'Sum':>6}")

for i, name in enumerate(token_names):
    print(f"  {name:>6}  ", end="")
    for j in range(seq_len):
        w = attn_weights[i, j].item()
        if w > 0.01:
            print(f"{w:8.3f}", end="")
        else:
            print(f"{'0':>8}", end="")
    print(f"  {attn_weights[i].sum().item():.3f}")

print(f"\n💡 Reading the table:")
print(f"   'cat' pays {attn_weights[1, 0]:.1%} attention to 'The' and {attn_weights[1, 1]:.1%} to itself.")
print(f"   'on' distributes attention across all 4 tokens it can see.")

In [ ]:
# === Step 5: Weighted Sum of Values ===
# Use the attention weights to create a weighted combination of Value vectors.
# Each token's output is a BLEND of information from the tokens it attends to.

print("=" * 60)
print("STEP 5: Weighted Sum of Values")
print("=" * 60)

# Compute: output = attention_weights @ V
output = attn_weights @ V  # (4, 4) @ (4, 3) → (4, 3)

print(f"\nFormula: output[i] = Σ_j  attn_weight[i,j] × V[j]")
print(f"\nLet's compute the output for 'cat' (position 1) by hand:")
print(f"  attn_weights[cat] = [{', '.join(f'{x:.3f}' for x in attn_weights[1].tolist())}]")
print(f"  V[The] = {V[0].tolist()}, V[cat] = {V[1].tolist()}")
print(f"  output[cat] = {attn_weights[1, 0]:.3f} × {V[0].tolist()} + {attn_weights[1, 1]:.3f} × {V[1].tolist()}")
manual = attn_weights[1, 0] * V[0] + attn_weights[1, 1] * V[1]
print(f"             = [{', '.join(f'{x:.3f}' for x in manual.tolist())}]")
print(f"  Actual:      [{', '.join(f'{x:.3f}' for x in output[1].tolist())}] ✅")

print(f"\nAll outputs:")
for i, name in enumerate(token_names):
    print(f"  {name:>4}: [{', '.join(f'{x:.3f}' for x in output[i].tolist())}]")

print(f"\n💡 Each token's output now contains information from all tokens")
print(f"   it attended to, blended according to the attention weights.")
print(f"   This is how context flows between tokens in a transformer!")

In [ ]:
# === Visualize all 5 steps together ===

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Plot 1: Raw attention scores
im0 = axes[0].imshow(raw_scores.detach().numpy(), cmap='RdBu_r', aspect='equal')
axes[0].set_xticks(range(seq_len)); axes[0].set_xticklabels(token_names)
axes[0].set_yticks(range(seq_len)); axes[0].set_yticklabels(token_names)
axes[0].set_title('1. Raw Scores (Q·K$^T$)', fontsize=11)
axes[0].set_xlabel('Key →'); axes[0].set_ylabel('← Query')
for i in range(seq_len):
    for j in range(seq_len):
        axes[0].text(j, i, f'{raw_scores[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Plot 2: After causal mask
display_masked = masked_scores.clone()
display_masked[causal_mask] = 0  # For display, use 0 instead of -inf
im1 = axes[1].imshow(display_masked.detach().numpy(), cmap='RdBu_r', aspect='equal')
axes[1].set_xticks(range(seq_len)); axes[1].set_xticklabels(token_names)
axes[1].set_yticks(range(seq_len)); axes[1].set_yticklabels(token_names)
axes[1].set_title('2. After Causal Mask', fontsize=11)
axes[1].set_xlabel('Key →')
for i in range(seq_len):
    for j in range(seq_len):
        if causal_mask[i, j]:
            axes[1].text(j, i, '✗', ha='center', va='center', fontsize=12, color='red')
        else:
            axes[1].text(j, i, f'{scaled_scores[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Plot 3: Attention weights (after softmax)
im2 = axes[2].imshow(attn_weights.detach().numpy(), cmap='Blues', aspect='equal', vmin=0, vmax=1)
axes[2].set_xticks(range(seq_len)); axes[2].set_xticklabels(token_names)
axes[2].set_yticks(range(seq_len)); axes[2].set_yticklabels(token_names)
axes[2].set_title('3. Attention Weights (softmax)', fontsize=11)
axes[2].set_xlabel('Key →')
for i in range(seq_len):
    for j in range(seq_len):
        w = attn_weights[i, j].item()
        color = 'white' if w > 0.5 else 'black'
        axes[2].text(j, i, f'{w:.2f}', ha='center', va='center', fontsize=8, color=color)
plt.colorbar(im2, ax=axes[2], fraction=0.046)

# Plot 4: Output values
im3 = axes[3].imshow(output.detach().numpy(), cmap='viridis', aspect='auto')
axes[3].set_yticks(range(seq_len)); axes[3].set_yticklabels(token_names)
axes[3].set_xticks(range(d_head)); axes[3].set_xticklabels([f'd{i}' for i in range(d_head)])
axes[3].set_title('4. Output (weights × V)', fontsize=11)
axes[3].set_xlabel('Dimension')
for i in range(seq_len):
    for j in range(d_head):
        axes[3].text(j, i, f'{output[i,j]:.2f}', ha='center', va='center', fontsize=8, color='white')
plt.colorbar(im3, ax=axes[3], fraction=0.046)

plt.suptitle('Self-Attention Step by Step', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../checkpoints/attention_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print("🔑 KEY TAKEAWAY:")
print("   The lower-left triangle of the attention weights shows the causal pattern.")
print("   Each token can only attend to itself and previous tokens.")
print("   This is what makes the model autoregressive (generates left-to-right).")

### 3.1 Multi-Head Attention: Multiple Perspectives

One attention head can only learn **one attention pattern** (e.g., "attend to the subject"). But language has many types of relationships!

**Multi-head attention** runs multiple attention computations in parallel:
- Head 1 might learn: "attend to the subject of the sentence"
- Head 2 might learn: "attend to the most recent noun"
- Head 3 might learn: "attend to the syntactic parent"
- Head 4 might learn: "attend to semantically similar words"

Instead of one attention with `d_model=128`, we use `n_heads=4` heads each with `d_head=32`. The results are concatenated and projected back to `d_model=128`.

Let's see this with the actual model's attention layer:

In [ ]:
# === Real Multi-Head Attention from our model ===
from model import CausalSelfAttention

attn_layer = CausalSelfAttention(cfg)

# Count parameters in the attention layer
n_params = sum(p.numel() for p in attn_layer.parameters())
print(f"=== Multi-Head Attention Layer ===")
print(f"  n_heads:  {cfg.n_heads} (parallel attention computations)")
print(f"  d_model:  {cfg.d_model} (total dimension)")
print(f"  d_head:   {cfg.d_model // cfg.n_heads} (dimension per head = d_model / n_heads)")
print(f"  Parameters: {n_params:,}")
print(f"")
print(f"  Breakdown:")
print(f"    QKV projection: {cfg.d_model} → {3 * cfg.d_model} = {cfg.d_model * 3 * cfg.d_model:,} params")
print(f"    Output projection: {cfg.d_model} → {cfg.d_model} = {cfg.d_model * cfg.d_model:,} params")

# Run a forward pass
dummy_input = torch.randn(1, 6, cfg.d_model)  # batch=1, seq_len=6
output = attn_layer(dummy_input)
print(f"\n  Input shape:  {dummy_input.shape}  (batch=1, seq=6, d_model={cfg.d_model})")
print(f"  Output shape: {output.shape}  (same shape — attention doesn't change dimensions)")

print(f"\n💡 The attention layer:")
print(f"   1. Projects input to Q, K, V (×3 the width)")
print(f"   2. Splits into {cfg.n_heads} heads of {cfg.d_model // cfg.n_heads} dims each")
print(f"   3. Each head does independent attention")
print(f"   4. Concatenates all heads back together")
print(f"   5. Projects back to d_model={cfg.d_model}")

---
## 4. 🏗️ The Full Model

Now let's see how all the pieces fit together in our `SmallGPT` model.

### Architecture Summary
```
Token IDs (batch, seq_len)
    ↓
Token Embedding + Position Embedding → (batch, seq_len, 128)
    ↓
Transformer Block 0:  LayerNorm → Attention → +residual → LayerNorm → FFN → +residual
    ↓
Transformer Block 1:  same structure
    ↓
Transformer Block 2:  same structure
    ↓
Transformer Block 3:  same structure
    ↓
Final LayerNorm → Linear Head → (batch, seq_len, 4000) ← logits for each vocab word
```

In [ ]:
from model import SmallGPT

# Reset the random seed for reproducibility
torch.manual_seed(42)
model = SmallGPT(cfg)

# === Parameter breakdown by component ===
print("\n📊 Parameter Breakdown:")
print(f"{'Component':<30} {'Parameters':>12} {'Shape'}")
print("-" * 70)

token_emb_params = model.token_emb.weight.numel()
print(f"{'Token embeddings':<30} {token_emb_params:>12,}   ({cfg.vocab_size} × {cfg.d_model})")

pos_emb_params = model.pos_emb.embedding.weight.numel()
print(f"{'Position embeddings':<30} {pos_emb_params:>12,}   ({cfg.max_seq_len} × {cfg.d_model})")

for i, block in enumerate(model.blocks):
    block_params = sum(p.numel() for p in block.parameters())
    attn_params = sum(p.numel() for p in block.attn.parameters())
    ffn_params = sum(p.numel() for p in block.ffn.parameters())
    ln_params = sum(p.numel() for p in block.ln1.parameters()) + sum(p.numel() for p in block.ln2.parameters())
    print(f"{'Transformer Block ' + str(i):<30} {block_params:>12,}   (attn:{attn_params:,} + ffn:{ffn_params:,} + ln:{ln_params:,})")

final_ln_params = sum(p.numel() for p in model.ln_final.parameters())
print(f"{'Final LayerNorm':<30} {final_ln_params:>12,}   (2 × {cfg.d_model})")
print(f"{'LM Head (tied weights)':<30} {'(shared)':>12}   (uses token embedding weights)")
print("-" * 70)
print(f"{'TOTAL':<30} {model.count_parameters():>12,}")
print(f"\n💡 Weight tying saves {cfg.vocab_size * cfg.d_model:,} parameters!")
print(f"   The LM head reuses the token embedding matrix.")

In [ ]:
# === Trace a forward pass step by step ===
sample_text = "The cat sat on"
encoded = tokenizer.encode(sample_text)
input_ids = torch.tensor([encoded.ids])

print(f"=== Tracing a Forward Pass ===")
print(f"\nInput text: '{sample_text}'")
print(f"Token IDs:  {encoded.ids}")
print(f"Tokens:     {encoded.tokens}")
print(f"Input shape: {input_ids.shape}  (batch=1, seq_len={input_ids.shape[1]})")

# Step-by-step forward pass (manually for educational purposes)
with torch.no_grad():
    # Step 1: Token embedding
    tok_emb = model.token_emb(input_ids)
    print(f"\nStep 1 - Token embedding:    {tok_emb.shape}  (each token → {cfg.d_model}-dim vector)")
    
    # Step 2: Position embedding
    pos_emb = model.pos_emb(input_ids.shape[1])
    print(f"Step 2 - Position embedding: {pos_emb.shape}  (one vector per position)")
    
    # Step 3: Add them together
    x = tok_emb + pos_emb
    print(f"Step 3 - tok_emb + pos_emb:  {x.shape}  (token identity + position info)")
    
    # Step 4: Pass through transformer blocks
    for i, block in enumerate(model.blocks):
        x = block(x)
        print(f"Step 4.{i} - Transformer block {i}: {x.shape}  (attention + FFN + residuals)")
    
    # Step 5: Final LayerNorm
    x = model.ln_final(x)
    print(f"Step 5 - Final LayerNorm:    {x.shape}  (normalize for stable output)")
    
    # Step 6: LM Head (project to vocabulary)
    logits = model.lm_head(x)
    print(f"Step 6 - LM Head → logits:   {logits.shape}  (one score per vocab word per position)")

# What does the model predict for the next token?
last_logits = logits[0, -1, :]  # Logits for last position
probs = F.softmax(last_logits, dim=-1)
top5 = torch.topk(probs, 5)

print(f"\n--- Predictions for next token after '{sample_text}' ---")
print(f"(Model is UNTRAINED, so predictions are random!)")
for prob, idx in zip(top5.values, top5.indices):
    token = tokenizer.id_to_token(idx.item())
    print(f"  '{token}' — {prob.item():.4f} ({prob.item()*100:.1f}%)")

---
## 5. 📈 Training: Teaching the Model to Predict

### The Training Objective
Training is conceptually simple — we're playing a **next-word prediction game** millions of times:

```
Input:  [The, cat, sat, on, the]     → Model sees this
Target: [cat, sat, on, the, mat]     → Model should predict this

At each position:
  Given [The]              → should predict 'cat'
  Given [The, cat]         → should predict 'sat'
  Given [The, cat, sat]    → should predict 'on'
  Given [The, cat, sat, on] → should predict 'the'
  Given [The, cat, sat, on, the] → should predict 'mat'
```

### Cross-Entropy Loss
The loss function measures how "surprised" the model is by the actual next token:
- If the model assigns **high probability** to the correct token → **low loss** ✅
- If the model assigns **low probability** to the correct token → **high loss** ❌

Mathematically: `loss = -log(P(correct_token))`

Examples:
- Model predicts "cat" with probability 0.8 → loss = -log(0.8) = 0.22 (low, good!)
- Model predicts "cat" with probability 0.01 → loss = -log(0.01) = 4.6 (high, bad!)
- Random model (1/4000 chance) → loss = -log(1/4000) = 8.3 (very high!)

### Training with `train.py`
The full training is run via: `python train.py`

Let's look at the training results if available:

In [ ]:
import json, os

history_path = '../checkpoints/loss_history.json'
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    if history.get('train'):
        steps = [e['step'] for e in history['train']]
        losses = [e['loss'] for e in history['train']]
        ax.plot(steps, losses, label='Train Loss', alpha=0.7, color='steelblue', linewidth=2)
    
    if history.get('val'):
        steps = [e['step'] for e in history['val']]
        losses = [e['loss'] for e in history['val']]
        ax.plot(steps, losses, label='Val Loss', marker='o', color='orangered', linewidth=2)
    
    # Add reference lines
    ax.axhline(y=math.log(cfg.vocab_size), color='gray', linestyle='--', alpha=0.5, label=f'Random model (ln({cfg.vocab_size})={math.log(cfg.vocab_size):.1f})')
    
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Loss (Cross-Entropy)', fontsize=12)
    ax.set_title('Training Progress — Is the Model Learning?', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print interpretation
    if history.get('train'):
        initial_loss = history['train'][0]['loss']
        final_loss = history['train'][-1]['loss']
        print(f"\n📊 Training Summary:")
        print(f"   Initial loss: {initial_loss:.2f} (near random: {math.log(cfg.vocab_size):.1f})")
        print(f"   Final loss:   {final_loss:.2f}")
        print(f"   Improvement:  {initial_loss - final_loss:.2f} ({(1 - final_loss/initial_loss)*100:.0f}% reduction)")
        print(f"\n   The model went from random guessing to learning real patterns!")
else:
    print("❌ No training history found.")
    print("   Run 'python train.py' first to train the model!")
    print(f"\n   Expected: loss starts at ~{math.log(cfg.vocab_size):.1f} (random) and decreases to ~4.0")

---
## 6. ✍️ Text Generation: Making the Model Write

### How Autoregressive Generation Works
Generation is the reverse of training: instead of computing loss, we **sample from the model's predictions** to produce new text.

```
Prompt: "Once upon a"
  ↓ feed to model
Model predicts: {"time": 0.6, "day": 0.2, "hill": 0.05, ...}
  ↓ sample (e.g., pick "time")
Now: "Once upon a time"
  ↓ feed to model  
Model predicts: {",": 0.5, ".": 0.1, "there": 0.15, ...}
  ↓ sample (e.g., pick ",")
Now: "Once upon a time,"
  ... continue until <EOS> or max length ...
```

### Why Not Always Pick the Top Token? (Greedy vs Sampling)
**Greedy decoding** (always picking the most likely token) tends to produce repetitive, boring text. **Sampling** introduces randomness for more diverse and interesting output.

The key controls are:
- **Temperature**: lower = more confident/focused, higher = more creative/random
- **Top-k**: only consider the top k most likely tokens
- **Top-p**: only consider tokens covering the top p% of probability mass

In [ ]:
from generate import load_model, generate
from config import GenerationConfig

# Load the trained model (or untrained if no checkpoint exists)
model, tokenizer = load_model()
device = next(model.parameters()).device

print(f"\nModel loaded on device: {device}")
print(f"Ready for generation! 🚀")

In [ ]:
# === Compare different sampling strategies ===
prompt = "Once upon a time"
print(f"Prompt: '{prompt}'\n")

# Strategy 1: Greedy (deterministic, often repetitive)
print("=" * 60)
print("🔵 GREEDY DECODING")
print("   Always picks the most probable token.")
print("   Deterministic but often gets stuck in loops.")
print("=" * 60)
cfg_greedy = GenerationConfig(greedy=True, max_new_tokens=80)
print(generate(model, tokenizer, prompt, cfg_greedy, device))

print()

# Strategy 2: Temperature sampling at different levels
for temp in [0.3, 0.8, 1.5]:
    print("=" * 60)
    emoji = "❄️" if temp < 0.5 else ("🌡️" if temp < 1.2 else "🔥")
    desc = "cold/focused" if temp < 0.5 else ("balanced" if temp < 1.2 else "hot/creative")
    print(f"{emoji} TEMPERATURE = {temp} ({desc})")
    print("=" * 60)
    cfg_temp = GenerationConfig(temperature=temp, top_k=50, top_p=0.9, max_new_tokens=80)
    print(generate(model, tokenizer, prompt, cfg_temp, device))
    print()

In [ ]:
# === Visualize the effect of temperature on the probability distribution ===

# Get the model's predictions for a specific context
encoded = tokenizer.encode("The little dog")
input_ids = torch.tensor([encoded.ids], device=device)

with torch.no_grad():
    logits, _ = model(input_ids)

last_logits = logits[0, -1, :]  # Next-token logits

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
temps = [0.3, 1.0, 2.0]
temp_labels = ['T=0.3 (focused)', 'T=1.0 (standard)', 'T=2.0 (creative)']

for ax, temp, label in zip(axes, temps, temp_labels):
    probs = F.softmax(last_logits / temp, dim=-1)
    top_k = 15
    top_probs, top_indices = torch.topk(probs, top_k)
    tokens = [tokenizer.id_to_token(i.item()) for i in top_indices]
    
    bars = ax.barh(range(top_k), top_probs.cpu().numpy(), color='steelblue', alpha=0.8)
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_title(label, fontsize=12)
    ax.set_xlabel('Probability', fontsize=10)
    ax.invert_yaxis()
    ax.set_xlim(0, max(0.8, top_probs[0].item() * 1.1))
    
    # Add probability labels
    for bar, prob in zip(bars, top_probs):
        if prob.item() > 0.01:
            ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                   f'{prob.item():.3f}', va='center', fontsize=8)

plt.suptitle('Effect of Temperature on Next-Token Probabilities', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../checkpoints/temperature_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 Key observations:")
print("   T=0.3 (low):  Very peaked — model strongly favors one token")
print("   T=1.0 (normal): Moderate distribution — a few likely candidates")
print("   T=2.0 (high): Flat distribution — many tokens have similar probability")
print("\n   In practice, T=0.7-0.9 with top_p=0.9 gives the best results.")

---
## 7. 🚀 What's Next? Scaling Up and Beyond

Congratulations! You now understand the core architecture behind GPT, LLaMA, and other large language models. Our model is tiny (1.2M parameters, ~500 steps of training), but it uses **exactly the same principles** as models 100,000× larger.

### Immediate Next Steps

| What to Try | How | Expected Result |
|-------------|-----|----------------|
| **Train longer** | `max_steps: 500 → 2000` in config.py | Lower loss, more coherent text |
| **Bigger model** | `d_model: 128 → 256, n_layers: 4 → 6` | Better text quality, slower training |
| **More data** | Add more .txt files to `data/` | Less overfitting, more knowledge |
| **Different text** | Train on code, poetry, etc. | Model learns that domain's patterns |

### Advanced Topics to Explore

1. **Rotary Positional Embeddings (RoPE)** — used by LLaMA and modern models. Encodes relative position by rotating Q and K vectors. Better for long contexts.

2. **KV-Cache** — speeds up generation by caching Key and Value computations for previous tokens. Essential for practical deployment.

3. **Flash Attention** — a memory-efficient attention implementation that's much faster on GPUs by tiling the computation.

4. **Fine-tuning** — take a pre-trained model and adapt it to a specific task (e.g., question answering, summarization).

5. **RLHF (Reinforcement Learning from Human Feedback)** — how ChatGPT is made helpful and safe. Uses human preferences to fine-tune the model beyond next-token prediction.

### The Scaling Laws
Research has shown that language model performance improves predictably with:
- **More parameters** (model size)
- **More data** (dataset size)
- **More compute** (training time)

Our 1.2M model → GPT-2's 1.5B → GPT-3's 175B → GPT-4's ~1.8T... each step unlocks new capabilities (summarization, reasoning, coding, etc.).

Happy learning! 🎓✨